# LogSieve experiments, Week 14

This notebook is the experimental plan for LogSieve and the first measurements I trust enough to put in the proposal. LogSieve is a Ferragina-Manzini index (FM-index) over a security log. The index is the Burrows-Wheeler Transform (BWT) stored in a wavelet tree, plus a sampled suffix array. I compare count queries with a plain suffix array, Knuth-Morris-Pratt (KMP), a `str.find` loop, and a naive scan. The seed is 5300. Each timed call is repeated so I can report the mean, the sample standard deviation, the median, and the interquartile range (IQR). The collector is disabled during a timed call.

The sizes in this run are prefixes of the OpenSSH log at 2,000, 10,000, and 50,000 lines, when `data/raw/SSH.log` is available. Larger prefixes, from about 1 MB up to about 20 MB, are sketched at the end and are not run yet. If the full log is missing, the harness uses the committed 2,000 line sample and the bigger prefixes collapse to that file.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "sample" / "OpenSSH_2k.log").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from logsieve.bench import SEED, run_week14

print("repo", ROOT)
print("seed", SEED)
summary = run_week14(seed=SEED, repeats=5, quick=False, output_dir=ROOT / "results")
print("timing file", summary["timing_csv"])
print("charts", len(summary["charts"]))


## Setup and the machine

The cell above writes `results/environment.json` before it starts timing. I want the Python version, the operating system, the CPU, the RAM, the git commit, and the seed in one place so a later run on my own PC can be compared with this one instead of being mixed into it. The next cell prints that file. The fit of build time on log log axes is stored there too, and it is computed from the medians of this run.


In [ ]:
env = json.loads((ROOT / "results" / "environment.json").read_text())
shown = {key: env[key] for key in (
    "python_version", "implementation", "os", "cpu", "cpu_count", "ram_bytes",
    "git_commit", "git_dirty", "seed", "repeats", "warmup",
    "gc_disabled_during_timed_sections", "openssh_source", "line_counts",
    "locate_pattern", "loglog_fit",
) if key in env}
print(json.dumps(shown, indent=2))


## Datasets and the query workload

The logs are the Loghub collection (Zhu et al. 2023), Zenodo record 8196385. The committed samples are `OpenSSH_2k.log` and `Linux_2k.log`. The full OpenSSH log is the unpacked `SSH.tar.gz` archive. A synthetic control uses the same number of body bytes as each OpenSSH prefix, drawn from a small printable alphabet with the same seed recipe, so I can see whether repetition in the real log changes the wavelet size.

The workload mixes three kinds of pattern. Random slices of length 4, 8, 16, 32, and 64 are taken from the log body with `random.Random(seed + line_count)`. Fixed analyst strings are "Failed password", "Invalid user", "authentication failure", "error", and "Accepted", plus the first IPv4 address in the prefix and its three octet prefix when one exists. One absent string, "zzqxvnotpresent", is included so a miss is timed too. Aho-Corasick is timed on watch lists of 10, 50, and 100 strings drawn from the 2,000 line prefix. A list of 500 strings is a Week 15 run.


## Correctness

I do not time a method until it agrees with the naive scan. For every pattern in the workload the harness compares FM-index count, KMP count, the `str.find` count, and suffix array binary search with the naive count. When a pattern has at most 30 hits, it also compares the located positions. The run raises if any of those checks fail. The next cell repeats a small version of that check in the notebook so the agreement is visible without opening the harness.


In [ ]:
from logsieve.baselines import find_count, kmp_count, naive_count, naive_search
from logsieve.fm_index import FMIndex
from logsieve.ingest import load_log
from logsieve.suffix_array import build_suffix_array, suffix_array_count, suffix_array_search

log = load_log(ROOT / "data" / "sample" / "OpenSSH_2k.log", max_lines=80)
text = log.text
index = FMIndex(text, sample_rate=8)
sa = build_suffix_array(text)
patterns = [b"Invalid user", b"error", b"zzqxvnotpresent", text[20:28], b""]
rows = []
for pattern in patterns:
    naive_hits = naive_search(text, pattern)
    row = {
        "pattern": pattern.decode("utf-8", "backslashreplace")[:40],
        "naive": len(naive_hits),
        "kmp": kmp_count(text, pattern),
        "str_find": find_count(text, pattern),
        "sa": suffix_array_count(text, sa, pattern),
        "fm": index.count(pattern),
        "locate_matches": index.locate(pattern) == naive_hits,
    }
    rows.append(row)
pd.DataFrame(rows)


## E1, build cost

For each prefix I build the FM-index five times and record the inner clocks for the suffix array, the BWT, the wavelet tree, the samples, and the total. The plot uses the median of those five trials on log log axes. A fitted slope is a description of these three sizes, not a proof of the asymptotic bound. Week 15 adds more sizes and a faster suffix array construction.


In [ ]:
summary_df = pd.read_csv(ROOT / "results" / "early_benchmarks_summary.csv")
build = summary_df[summary_df.experiment == "e1_build"].sort_values(["method", "lines"])
cols = ["method", "lines", "text_bytes", "trials", "mean_seconds", "std_seconds", "median_seconds", "iqr_seconds"]
build[cols]


In [ ]:
display(Image(filename=str(ROOT / "results" / "e1_build_time.png")))
print("log log slope for fm_total", env["loglog_fit"].get("fm_total_loglog_slope"))


## E2, space

Space is measured once per text, because the structures are deterministic. I record the raw text, a numpy int32 suffix array, the same array stored as a Python list of integers (object headers included), the packed wavelet bits, the rank directory, the suffix array samples, and the retained BWT array. `fm_index_bytes` adds those index pieces and does not include the original text. `bits_per_char` is the number of wavelet bit vector entries divided by the text length, which is the bit cost of the tree before packing and before the rank directory. The synthetic rows are the random controls at the same body length. The Linux row is the committed 2,000 line Linux sample.


In [ ]:
space = pd.read_csv(ROOT / "results" / "space_measurements.csv")
keep = ["dataset", "lines", "text_bytes", "sa_numpy_bytes", "sa_python_bytes",
        "wavelet_packed_bytes", "rank_directory_bytes", "wavelet_bits", "bits_per_char",
        "sample_bytes", "bwt_bytes", "fm_index_bytes", "fm_without_raw_bwt_bytes", "distinct_symbols"]
space[keep]


In [ ]:
display(Image(filename=str(ROOT / "results" / "e2_space.png")))


## E3, count query latency

Count queries are timed after the index and the suffix array already exist, so the build is not mixed into the query. The naive scan, KMP, and `str.find` only count. They do not build a hit list. The size plot is the random length 8 pattern. The length plot is the random patterns on the largest prefix. Analyst strings and the absent string are in the CSV with `pattern_kind` set, and they are not on those two lines. Median and IQR are in the summary table. Mean and standard deviation are the variance I want next to the median, because a single average would hide a slow trial.


In [ ]:
counts = summary_df[summary_df.experiment == "e3_count"]
view = counts[counts.pattern_kind.isin(["random", "analyst", "absent", "ip"])]
view.groupby(["lines", "method", "pattern_kind", "pattern_len"], as_index=False)[
    ["mean_seconds", "std_seconds", "median_seconds", "iqr_seconds", "matches"]
].median(numeric_only=True)


In [ ]:
display(Image(filename=str(ROOT / "results" / "e3_latency_vs_size.png")))
display(Image(filename=str(ROOT / "results" / "e3_latency_vs_length.png")))


## E4, sample rate, small scale

Locate walks the LF mapping until it hits a sampled text position. The sample rate k is the gap between stored positions, so a larger k stores less and walks farther. This Week 14 curve is on the 2,000 line prefix only. Rebuilding a 50,000 line index once per sample rate is Week 15 work, after the suffix array build is cheaper. The pattern is chosen in the harness as the busiest workload string that still has at most 80 hits, and its text is stored in `environment.json` under `locate_pattern`. The right axis of the chart is the measured sample bytes, not a formula.


In [ ]:
locate = summary_df[summary_df.experiment == "e4_locate"].sort_values("sample_rate")
locate[["sample_rate", "pattern", "matches", "mean_seconds", "std_seconds", "median_seconds", "iqr_seconds"]]


In [ ]:
memory = pd.read_csv(ROOT / "results" / "sample_rate_memory.csv")
memory


In [ ]:
display(Image(filename=str(ROOT / "results" / "e4_sample_tradeoff.png")))


## E5, many patterns, small scale

One Aho-Corasick pass over the 2,000 line prefix is compared with repeating KMP and with repeating FM-index counts, for 10, 50, and 100 patterns. The patterns are random length 8 slices. A watch list of 500, and the same comparison on a much longer log, is the Week 15 version of this experiment. The question I want answered there is where the index pays for its build when the analyst already has a batch of patterns and a single online pass can find all of them.


In [ ]:
watch = summary_df[summary_df.experiment == "e5_watch"].sort_values(["matches", "method"])
watch[["method", "matches", "lines", "mean_seconds", "std_seconds", "median_seconds", "iqr_seconds"]]


In [ ]:
display(Image(filename=str(ROOT / "results" / "e5_many_patterns.png")))


## E6, Count-Min sketch, stretch

The Count-Min sketch estimates how often each IPv4 address appears. It never returns a count below the true count. I compare it with `collections.Counter` on the largest prefix in this run. Width changes and depth stays at 4. The exact counter's Python object size is measured with `sys.getsizeof` on the counter and on each key. This is a small sweep. A wider error curve, and a top talker list that does not start from the exact key set, belongs to Week 15.


In [ ]:
sketch = pd.read_csv(ROOT / "results" / "sketch_measurements.csv")
sketch


In [ ]:
display(Image(filename=str(ROOT / "results" / "e6_sketch_error.png")))


## Break even

Amakobe section 14.3 is the reason this table exists. The build has to be paid for by later queries. I take the median FM-index build time and compare the median `str.find` count with the median FM-index count across the workload at that size. If the index is faster per query, the number of queries that pay for the build is the build time divided by that gap. If the index is slower per query, the status column says so and the crossover cell is left blank. The medians that go into the division are the medians of the per pattern medians, so one very common string does not decide the whole size by itself.


In [ ]:
pd.read_csv(ROOT / "results" / "breakeven.csv")


## Threats to validity, and what Week 15 still has to run

These numbers are from one machine, the one named in `environment.json`. A second machine, including the Windows PC I will use for the final numbers, can move the constant factors around even when the ordering of the methods stays put. The collector is off during timed calls, and there is one warmup, but the file cache and the CPU frequency are not pinned. Python object overhead is part of the space result. The rank directory and the Python suffix array list are much larger than the textbook bit counts, and I report both the wavelet bit entries and the bytes actually allocated.

The naive scan in this harness skips ahead with `bytes.find` on the first byte of the pattern and then checks the rest. That is still a linear scan, and its worst case is still proportional to the product of the lengths. It is a C assisted scan, while KMP here is a pure Python loop, so a slow KMP time is partly the interpreter. The report will call that gap an implementation effect.

Week 15 still has to run the prefixes at 1, 2, 5, 10, and 20 MB, a suffix array construction that is closer to linear time, locate on the larger prefixes, a watch list of 500 patterns, and a Count-Min sweep that chooses its own candidates. The cell below only records that plan. It does not build those indexes.


In [ ]:
def week15_plan():
    """Skeleton for the runs that are too heavy for this pass."""
    return {
        "prefixes_mb": [1, 2, 5, 10, 20],
        "suffix_array": "radix doubling or SA-IS, still checked against this builder",
        "locate_lines": [10000, 50000],
        "watch_sizes": [10, 50, 100, 500],
        "sketch": "sweep width and depth, and keep a candidate heap",
    }

print("planned for Week 15")
for key, value in week15_plan().items():
    print(key, value)
